# Prepare 80/20 training and validation datasets

This notebook splits `keypoints_train_filtered_subposes.csv` independently within every original yoga pose. It targets 80% training and 20% validation using the nearest possible whole-row count. Retained subpose labels are also stratified within each pose when possible.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20
POSE_COLUMN = 'label'
SUBPOSE_COLUMN = 'subpose_label'


def find_project_root(start: Path = Path.cwd()) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        candidate = (
            directory
            / 'csv_data'
            / 'keypoint_filtered_subposes'
            / 'keypoints_train_filtered_subposes.csv'
        )
        if candidate.is_file():
            return directory
    raise FileNotFoundError('Could not find the filtered training CSV.')


PROJECT_ROOT = find_project_root()
INPUT_FILE = (
    PROJECT_ROOT
    / 'csv_data'
    / 'keypoint_filtered_subposes'
    / 'keypoints_train_filtered_subposes.csv'
)
OUTPUT_DIR = PROJECT_ROOT / 'csv_data' / 'prepared_to_train'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT = OUTPUT_DIR / 'keypoints_train.csv'
VALIDATION_OUTPUT = OUTPUT_DIR / 'keypoints_validation.csv'
POSE_SUMMARY_OUTPUT = OUTPUT_DIR / 'pose_split_summary.csv'
SUBPOSE_SUMMARY_OUTPUT = OUTPUT_DIR / 'subpose_split_summary.csv'

PROJECT_ROOT

WindowsPath('C:/Users/vgohu/Desktop/yoga_project')

In [2]:
source = pd.read_csv(INPUT_FILE)
required_columns = {POSE_COLUMN, SUBPOSE_COLUMN}
missing_columns = required_columns.difference(source.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
if source.iloc[:, 0].duplicated().any():
    raise ValueError('The first-column image paths must be unique before splitting.')

training_parts = []
validation_parts = []
split_details = []

for pose_number, pose in enumerate(sorted(source[POSE_COLUMN].unique())):
    pose_rows = source[source[POSE_COLUMN].eq(pose)]
    validation_count = round(len(pose_rows) * VALIDATION_FRACTION)
    validation_count = max(1, min(validation_count, len(pose_rows) - 1))

    subpose_counts = pose_rows[SUBPOSE_COLUMN].value_counts()
    can_stratify = (
        len(subpose_counts) > 1
        and subpose_counts.min() >= 2
        and validation_count >= len(subpose_counts)
        and (len(pose_rows) - validation_count) >= len(subpose_counts)
    )
    stratify_values = pose_rows[SUBPOSE_COLUMN] if can_stratify else None

    pose_train, pose_validation = train_test_split(
        pose_rows,
        test_size=validation_count,
        random_state=RANDOM_STATE + pose_number,
        shuffle=True,
        stratify=stratify_values,
    )
    training_parts.append(pose_train)
    validation_parts.append(pose_validation)
    split_details.append({
        'pose': pose,
        'total': len(pose_rows),
        'training': len(pose_train),
        'validation': len(pose_validation),
        'training_percent': 100 * len(pose_train) / len(pose_rows),
        'validation_percent': 100 * len(pose_validation) / len(pose_rows),
        'subpose_stratified': can_stratify,
    })

training = (
    pd.concat(training_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
validation = (
    pd.concat(validation_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
pose_summary = pd.DataFrame(split_details)
pose_summary

,pose,total,training,validation,training_percent,validation_percent,subpose_stratified
0,downdog,199,159,40,79.899497,20.100503,False
1,goddess,162,130,32,80.246914,19.753086,False
2,plank,255,204,51,80.000000,20.000000,True
3,tree,150,120,30,80.000000,20.000000,True
4,warrior2,249,199,50,79.919679,20.080321,True


In [3]:
training.to_csv(TRAIN_OUTPUT, index=False)
validation.to_csv(VALIDATION_OUTPUT, index=False)
pose_summary.to_csv(POSE_SUMMARY_OUTPUT, index=False)

subpose_summary = (
    pd.concat(
        [
            source[SUBPOSE_COLUMN].value_counts().rename('total'),
            training[SUBPOSE_COLUMN].value_counts().rename('training'),
            validation[SUBPOSE_COLUMN].value_counts().rename('validation'),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .rename_axis('subpose_label')
    .reset_index()
    .sort_values('subpose_label')
)
subpose_summary['training_percent'] = 100 * subpose_summary['training'] / subpose_summary['total']
subpose_summary['validation_percent'] = 100 * subpose_summary['validation'] / subpose_summary['total']
subpose_summary.to_csv(SUBPOSE_SUMMARY_OUTPUT, index=False)
subpose_summary

,subpose_label,total,training,validation,training_percent,validation_percent
0,downdog_subpose_1,199,159,40,79.899497,20.100503
1,goddess_subpose_2,162,130,32,80.246914,19.753086
8,plank_subpose_1,13,10,3,76.923077,23.076923
3,plank_subpose_2,124,99,25,79.838710,20.161290
4,plank_subpose_4,118,95,23,80.508475,19.491525
7,tree_left_subpose_2,50,40,10,80.000000,20.000000
5,tree_right_subpose_2,100,80,20,80.000000,20.000000
6,warrior2_left_subpose_1,87,70,17,80.459770,19.540230
2,warrior2_right_subpose_1,162,129,33,79.629630,20.370370


In [4]:
path_column = source.columns[0]
training_paths = set(training[path_column])
validation_paths = set(validation[path_column])
source_paths = set(source[path_column])

assert training_paths.isdisjoint(validation_paths)
assert training_paths | validation_paths == source_paths
assert len(training) + len(validation) == len(source)
assert list(training.columns) == list(source.columns) == list(validation.columns)
assert TRAIN_OUTPUT.is_file() and VALIDATION_OUTPUT.is_file()
assert POSE_SUMMARY_OUTPUT.is_file() and SUBPOSE_SUMMARY_OUTPUT.is_file()
for row in split_details:
    assert row['training'] + row['validation'] == row['total']
    assert abs(row['validation'] - round(row['total'] * VALIDATION_FRACTION)) <= 1

print(f'Total source rows: {len(source):,}')
print(f'Training rows:     {len(training):,} ({len(training) / len(source):.2%})')
print(f'Validation rows:   {len(validation):,} ({len(validation) / len(source):.2%})')

Total source rows: 1,015
Training rows:     812 (80.00%)
Validation rows:   203 (20.00%)


## Outputs

Files are saved in `csv_data/prepared_to_train/`:

- `keypoints_train.csv`
- `keypoints_validation.csv`
- `pose_split_summary.csv`
- `subpose_split_summary.csv`